In [7]:
%reset -f
import arcpy
arcpy.ResetEnvironments()
import os
import csv

In [8]:
# ── GS METHOD: GEODATABASE PATHS ─────────────────────────────────────────────────────────────────────

# Lists of geodatabases containing feature classes whose properties will be written to each output CSV - each gdb outputs one CSV with each feature class in the gdb as a row in CSV and each feature class property as a column; outputs CSVs named by gdb name (i.e., "Caribbean_Shorelines_raw.gdb" gdb -> Caribbean_Shorelines_raw.csv)
gdb_list = [
    r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\GEBCO_Gridded2025_15arcsec.gdb",
    r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\Jeremiah_NOAA_10meter_USCaribbean.gdb",
    r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Bathymetry_geodatabases\MASTER_RASTERS_USC_GBC_MERGED.gdb",
    r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Shoreline_geodatabases\Caribbean_shorelines.gdb",
    r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Geodatabases\Thesis_geodatabases\MASTER_THESIS.gdb",
    r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\Raw_data\Caribbean_Shorelines_raw.gdb"
]

# Folder to save output CSVs to (each CSV will be named based on the geodatabase it contains info for)
csv_folder = r"C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\metadata_and_information_csvs\featureclass_information_csvs\csv_by_geodatabase"

In [9]:
# ── SETTINGS ─────────────────────────────────────────────────────────────────────────────────────────
arcpy.env.overwriteOutput = True    # overwrite existing outputs on rerun

In [10]:
# ── DICTIONARY OF FEATURE CLASS PROPERTIES TO INCLUDE IN CSV ─────────────────────────────────────────

def get_feature_class_properties(fc_path):
    # get the Describe object (contains name, shape type, spatial reference, extent, etc.)
    desc = arcpy.Describe(fc_path)
    # get feature count (number of rows/features in the feature class)
    feature_count = int(arcpy.management.GetCount(fc_path)[0])
    # build dictionary mapping each property label to its value
    return {
        # store input path so you can trace each feature class back to source
        "Path": fc_path,
        # dataset name without full path
        "Name": desc.name,
        # feature class type (e.g. Simple)
        "FeatureType": desc.featureType,
        # geometry type (e.g. Point, Polyline, Polygon)
        "ShapeType": desc.shapeType,
        # number of features
        "FeatureCount": feature_count,
        # number of fields in the attribute table
        "FieldCount": len(desc.fields),
        # comma-separated list of field names
        "Fields": ", ".join(f.name for f in desc.fields),
        # name of the geometry field (usually Shape)
        "ShapeFieldName": desc.shapeFieldName,
        # name of the Object ID field (usually OBJECTID)
        "OIDFieldName": desc.OIDFieldName,
        # whether geometry has Z (elevation) values
        "HasZ": desc.hasZ,
        # whether geometry has M (measure) values
        "HasM": desc.hasM,
        # name of coordinate system (projected name if projected, else geographic)
        "SpatialRef": desc.spatialReference.name,
        # geographic coordinate system name (underlying datum/GCS)
        "GeogCS": desc.spatialReference.GCS.name if desc.spatialReference.GCS else None,
        # vertical coordinate system name (returns "None" string if feature class has no vertical CS defined)
        "VertCS": desc.spatialReference.VCS.name if desc.spatialReference.VCS and desc.spatialReference.VCS.name else None,
        # spatial extent (xmin, ymin, xmax, ymax) columns
        "Extent_Xmin": desc.extent.XMin,
        "Extent_Ymin": desc.extent.YMin,
        "Extent_Xmax": desc.extent.XMax,
        "Extent_Ymax": desc.extent.YMax,
    }

In [11]:
# ── COMPILE FEATURE CLASS PROPERTIES INTO CSV DATA ───────────────────────────────────────────────────

# Function to write feature class properties to CSV (1 feature class per row, 1 property per column)
def export_feature_class_properties(feature_classes, out_csv):
    # if single path string was passed, wrap it in list so loop works
    if isinstance(feature_classes, str):
        feature_classes = [feature_classes]

    # list that will hold one dictionary (row) per feature class
    rows = []
    # loop over every feature class path provided
    for fc in feature_classes:
        try:
            # collect properties of feature class and add them as a single row
            rows.append(get_feature_class_properties(fc))
        # print warning if feature class fails and continue to next feature class in gdb
        except Exception as e:
            print(f"❌[FAILURE] CSV data failed to write for feature class {os.path.basename(fc)}: {e} \n")

    # if nothing was collected, warn and stop before writing empty file
    if not rows:
        print(f"⚠️[WARNING] No properties collected (empty CSV file): {os.path.basename(out_csv)} \n")
        return

    # use keys from the first row as CSV column headers
    fieldnames = list(rows[0].keys())
    # open output CSV for writing (newline="" avoids blank lines on Windows)
    with open(out_csv, "w", newline="") as f:
        # create writer that maps dictionaries to CSV columns
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        # write header row
        writer.writeheader()
        # write one row per feature class dictionary
        writer.writerows(rows)

    # print message confirming how many feature classes were written and to where
    print(f"✅ Wrote {len(rows)} feature class(es) to CSV: {out_csv} \n")

In [12]:
# ── GS METHOD: CALL FUNCTION AND EXPORT CSVs (1 CSV PER GEODATABASE) ─────────────────────────────────

# Loop over each geodatabase and all feature classes in it
for gdb in gdb_list:
    # point workspace at current gdb so ListFeatureClasses looks inside it
    arcpy.env.workspace = gdb
    # build full path to every feature class in this gdb (root + feature datasets in root gdb)
    fc_list = []
    # add feature classes at gdb root
    for fc in arcpy.ListFeatureClasses():
        fc_list.append(os.path.join(gdb, fc))
    # add feature classes inside each feature dataset
    for fds in arcpy.ListDatasets(feature_type="Feature"):
        for fc in arcpy.ListFeatureClasses(feature_dataset=fds):
            fc_list.append(os.path.join(gdb, fds, fc))
    # name CSV after gdb (e.g. Whale_Sightings.csv)
    gdb_name = os.path.splitext(os.path.basename(gdb))[0]
    out_csv = os.path.join(csv_folder, f"{gdb_name}.csv")
    # call function to write one combined CSV for all feature classes in gdb
    export_feature_class_properties(fc_list, out_csv)

✅ Wrote 1 feature class(es) to CSV: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\metadata_and_information_csvs\featureclass_information_csvs\csv_by_geodatabase\GEBCO_Gridded2025_15arcsec.csv 

✅ Wrote 4 feature class(es) to CSV: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\metadata_and_information_csvs\featureclass_information_csvs\csv_by_geodatabase\Jeremiah_NOAA_10meter_USCaribbean.csv 

✅ Wrote 6 feature class(es) to CSV: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\metadata_and_information_csvs\featureclass_information_csvs\csv_by_geodatabase\MASTER_RASTERS_USC_GBC_MERGED.csv 

✅ Wrote 3 feature class(es) to CSV: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\metadata_and_information_csvs\featureclass_information_csvs\csv_by_geodatabase\Caribbean_shorelines.csv 

✅ Wrote 1 feature class(es) to CSV: C:\Users\kkeii\Documents\ArcGIS\Files\THESIS_FILES\metadata_and_information_csvs\featureclass_information_csvs\csv_by_geodatabase\MASTER_THESIS.csv 

✅ Wrote 1 feat